In [1]:
import ee
import pandas as pd

ee.Initialize(project='ee-chowdharypranav2005')

districts = pd.read_csv(
    "../data/raw/outbreaks/district_master.csv"
)

print("Loaded", len(districts), "districts")

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

In [6]:
forest = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")

In [7]:
def extract_forest_loss(lat, lon, buffer_radius=50000):

    region = ee.Geometry.Point([lon, lat]).buffer(buffer_radius)

    loss = forest.select("loss")

    stats = loss.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=30,
        maxPixels=1e9
    )

    return stats.get("loss").getInfo()

In [8]:
value = extract_forest_loss(
    lat=13.9299,
    lon=75.5681
)

print(value)

0.019143822343316123


In [9]:
records = []

for _, district in districts.iterrows():

    value = extract_forest_loss(
        district["Latitude"],
        district["Longitude"]
    )

    records.append({
        "State": district["State"],
        "District": district["District"],
        "Forest_Loss": value
    })

forest_df = pd.DataFrame(records)

forest_df.head()

,State,District,Forest_Loss
0,Karnataka,Shivamogga,0.019144
1,Karnataka,Uttara Kannada,0.008988
2,Karnataka,Chikkamagaluru,0.018034
3,Karnataka,Kodagu,0.021077
4,Kerala,Wayanad,0.016563


In [10]:
forest_df.to_csv(
    "../data/processed/forest_loss_western_ghats.csv",
    index=False
)

print("Forest Loss Dataset Saved!")

Forest Loss Dataset Saved!
